In [11]:
#   ntp_info Check:
import uproot
import pandas as pd

file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
with uproot.open(file_path) as f:
    info_df = f["ntp_info"].arrays(library="pd")

# Filter for the same event you analyzed
# event_1_info = info_df[info_df.event == 1].iloc[0]

# print("--- Event 1 Global Metadata ---")
# print(f"Total Tracks in Event:    {event_1_info.ntrk}")
# print(f"Total TPC Hits:           {event_1_info.nhittpcall}")
# print(f"Total TPC Clusters:       {event_1_info.nclustpc}")
# print(f"MVTX (Silicon) Hits:      {event_1_info.nhitmvtx}")
# print(f"Centrality (MBD Raw):     {event_1_info.rawmbd}")
for row in info_df.itertuples(index=False):
    print(f"--- Event {int(row.event)} Global Metadata ---")
    print(f"Total Tracks in Event:    {row.ntrk}")
    print(f"Total TPC Hits:           {row.nhittpcall}")
    print(f"Total TPC Clusters:       {row.nclustpc}")
    print(f"MVTX (Silicon) Hits:      {row.nhitmvtx}")
    print(f"Centrality (MBD Raw):     {row.rawmbd}")
    print("-" * 35) # Adds a separator line between events

--- Event 1 Global Metadata ---
Total Tracks in Event:    26.0
Total TPC Hits:           56017.0
Total TPC Clusters:       6601.0
MVTX (Silicon) Hits:      14312.0
Centrality (MBD Raw):     13823.0
-----------------------------------
--- Event 2 Global Metadata ---
Total Tracks in Event:    72.0
Total TPC Hits:           195072.0
Total TPC Clusters:       22808.0
MVTX (Silicon) Hits:      13420.0
Centrality (MBD Raw):     13825.0
-----------------------------------
--- Event 3 Global Metadata ---
Total Tracks in Event:    65.0
Total TPC Hits:           121409.0
Total TPC Clusters:       13782.0
MVTX (Silicon) Hits:      15352.0
Centrality (MBD Raw):     13838.0
-----------------------------------
--- Event 4 Global Metadata ---
Total Tracks in Event:    47.0
Total TPC Hits:           121360.0
Total TPC Clusters:       14515.0
MVTX (Silicon) Hits:      13429.0
Centrality (MBD Raw):     13847.0
-----------------------------------
--- Event 5 Global Metadata ---
Total Tracks in Event:    

In [14]:
import uproot
import pandas as pd

file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
with uproot.open(file_path) as f:
    info_df = f["ntp_info"].arrays(library="pd")

# 1. Filter out 'Empty' events, 100 is simply a guess of threshold
useful_events = info_df[info_df.nhittpcall > 100]

# 2. Sort by complexity (Curriculum Learning)
easy_events = useful_events[useful_events.ntrk < 10].event.values
hard_events = useful_events[useful_events.ntrk > 50].event.values

# 3. Create a 'Global Context' vector for our model
# Example: Input = [hits_coords] + [occupancy_stats]
def get_global_context(ev_id):
    row = info_df[info_df.event == ev_id]
    return row[['ntrk', 'nhittpcall', 'occ11', 'rawmbd']].values

# === COMPLETE THE CHECK ===

# 1. Print the breakdown of your event curriculum
print("--- Event Complexity Breakdown ---")
print(f"Total events in ntp_info: {len(info_df)}")
print(f"Useful events (hits > 100): {len(useful_events)}")
print(f"Easy events (tracks < 10): {len(easy_events)}")
print(f"Hard events (tracks > 50): {len(hard_events)}")

# 2. Test the Global Context vector extraction
print("\n--- Testing Global Context Vector ---")
if len(useful_events) > 0:
    # # Pick the first available hard event to test (or a useful event if no hard ones exist)
    # test_ev_id = hard_events[0] if len(hard_events) > 0 else useful_events.iloc[0]['event']
    
    # # Extract the vector
    # context_vector = get_global_context(test_ev_id)
    
    # print(f"Extracted context for Event ID: {test_ev_id}")
    # print(f"Raw array shape: {context_vector.shape} (Ready for PyTorch tensors!)")
    
    # # Index [0] to get inside the 2D array wrapping
    # print(f"  -> ntrk (Tracks):        {context_vector[0][0]}")
    # print(f"  -> nhittpcall (Hits):    {context_vector[0][1]}")
    # print(f"  -> occ11 (Occupancy):    {context_vector[0][2]:.5f}")
    # print(f"  -> rawmbd (MBD Charge):  {context_vector[0][3]:.2f}")
    # Loop over every event ID in the useful_events DataFrame
    for ev_id in useful_events['event']:
        # Extract the vector
        context_vector = get_global_context(ev_id)
        
        print(f"Extracted context for Event ID: {ev_id}")
        print(f"Raw array shape: {context_vector.shape}")
        
        # Index [0] to get inside the 2D array wrapping
        print(f"  -> ntrk (Tracks):        {context_vector[0][0]}")
        print(f"  -> nhittpcall (Hits):    {context_vector[0][1]}")
        print(f"  -> occ11 (Occupancy):    {context_vector[0][2]:.5f}")
        print(f"  -> rawmbd (MBD Charge):  {context_vector[0][3]:.2f}")
        print("-" * 40) # Divider between events for readability
else:
    print("No useful events found to test.")

--- Event Complexity Breakdown ---
Total events in ntp_info: 100
Useful events (hits > 100): 100
Easy events (tracks < 10): 1
Hard events (tracks > 50): 66

--- Testing Global Context Vector ---
Extracted context for Event ID: 1.0
Raw array shape: (1, 4)
  -> ntrk (Tracks):        26.0
  -> nhittpcall (Hits):    56017.0
  -> occ11 (Occupancy):    0.00090
  -> rawmbd (MBD Charge):  13823.00
----------------------------------------
Extracted context for Event ID: 2.0
Raw array shape: (1, 4)
  -> ntrk (Tracks):        72.0
  -> nhittpcall (Hits):    195072.0
  -> occ11 (Occupancy):    0.00338
  -> rawmbd (MBD Charge):  13825.00
----------------------------------------
Extracted context for Event ID: 3.0
Raw array shape: (1, 4)
  -> ntrk (Tracks):        65.0
  -> nhittpcall (Hits):    121409.0
  -> occ11 (Occupancy):    0.00236
  -> rawmbd (MBD Charge):  13838.00
----------------------------------------
Extracted context for Event ID: 4.0
Raw array shape: (1, 4)
  -> ntrk (Tracks):       

In [3]:
import uproot
import numpy as np
import awkward as ak

file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
with uproot.open(file_path) as f:
    # 1. Load the hits and the event metadata
    hits = f["ntp_hit"].arrays(["event", "x", "y", "z", "hitID"], library="ak")
    info = f["ntp_info"].arrays(["event", "occ11", "occ21", "occ31"], library="ak")

# Example: Prepare data for Event 1
target_ev = 1
event_hits = hits[hits.event == target_ev]
event_info = info[info.event == target_ev]

# Create hit features: [x, y, z]
x_train = np.column_stack([ak.to_numpy(event_hits.x), ak.to_numpy(event_hits.y), ak.to_numpy(event_hits.z)])

# Create global context: [occ_inner, occ_mid, occ_outer]
# We repeat the global info for every hit in the event
occ_vector = np.array([event_info.occ11[0], event_info.occ21[0], event_info.occ31[0]])
global_features = np.tile(occ_vector, (len(x_train), 1))

# Final Training Input: [x, y, z, occ_i, occ_m, occ_o]
final_input = np.hstack([x_train, global_features])

print(f"Input shape for Event {target_ev}: {final_input.shape}")
# Result: (N_hits, 6)

Input shape for Event 1: (72651, 6)


In [16]:
import uproot
import pandas as pd
import numpy as np

file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
with uproot.open(file_path) as root_file:
    df_clus = root_file["ntp_cluster"].arrays(["event", "layer", "x", "y", "z"], library="pd")
    df_trk = root_file["ntp_clus_trk"].arrays(["event", "layer", "x", "y", "z", "spt"], library="pd")

# Filter down to a single event (e.g., Event 74) to make the math easy to see
event_id = 74
df_clus_ev = df_clus[df_clus['event'] == event_id].copy()
df_trk_ev = df_trk[df_trk['event'] == event_id].copy()

print(f"\n--- DATASET SIZES FOR EVENT {event_id} ---")
print(f"Total Raw Clusters found by Island Alg : {len(df_clus_ev):,}")
print(f"Clusters assigned to valid Tracks      : {len(df_trk_ev):,}")

# --- ML Labeling ---
merge_keys = ['event', 'layer', 'x', 'y', 'z']
df_clus_ev[merge_keys] = df_clus_ev[merge_keys].round(4)
df_trk_ev[merge_keys] = df_trk_ev[merge_keys].round(4)

# Perform the merge. The 'indicator=True' flag tells us if the cluster exists in BOTH trees.
df_labeled = pd.merge(df_clus_ev, df_trk_ev, on=merge_keys, how='left', indicator=True)

# Create the Machine Learning Target Label
# If the hit exists in 'both', it is part of a real track (Signal = 1)
# If the hit only exists in 'left_only', it is background noise (Signal = 0)
df_labeled['is_signal'] = np.where(df_labeled['_merge'] == 'both', 1, 0)

# Drop the temporary merge indicator
df_labeled = df_labeled.drop(columns=['_merge'])

# --- VERIFICATION RESULTS ---
total_hits = len(df_labeled)
signal_hits = df_labeled['is_signal'].sum()
noise_hits = total_hits - signal_hits
purity = (signal_hits / total_hits) * 100

print("\n--- MACHINE LEARNING TARGET VERIFICATION ---")
print(f"Dataset Successfully Labeled!")
print(f"May Be Signal Hits (is_signal = 1) : {signal_hits:,} ({purity:.2f}%) -> Contains 'spt' momentum features")
print(f"May Be Noise Hits  (is_signal = 0) : {noise_hits:,} ({(100-purity):.2f}%) -> 'spt' is NaN")

print("\nSample of your newly labeled Training Data:")
print(df_labeled[['layer', 'x', 'y', 'z', 'spt', 'is_signal']].head(10000))


--- DATASET SIZES FOR EVENT 74 ---
Total Raw Clusters found by Island Alg : 46,755
Clusters assigned to valid Tracks      : 3,292

--- MACHINE LEARNING TARGET VERIFICATION ---
Dataset Successfully Labeled!
May Be Signal Hits (is_signal = 1) : 3,292 (7.04%) -> Contains 'spt' momentum features
May Be Noise Hits  (is_signal = 0) : 43,468 (92.96%) -> 'spt' is NaN

Sample of your newly labeled Training Data:
      layer          x       y           z  spt  is_signal
0       0.0   2.959600  0.7342   -9.080500  NaN          0
1       0.0   3.347800 -0.0286   -9.576800  NaN          0
2       0.0   2.959400  0.7347   -9.080700  NaN          0
3       0.0   3.347800 -0.0286   -9.576800  NaN          0
4       0.0   3.335600 -0.0046  -10.062200  NaN          0
...     ...        ...     ...         ...  ...        ...
9995   14.0  34.669800 -7.4517   43.752499  NaN          0
9996   14.0  34.948002 -6.0222 -195.199997  NaN          0
9997   14.0  34.469200  8.3132  -24.414000  NaN          0
99

In [18]:
# import uproot
# import pandas as pd
# import numpy as np

# file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
# tree_hit = uproot.open(file_path)["ntp_hit"]
# tree_trk = uproot.open(file_path)["ntp_clus_trk"]

# # We use the discrete detector pad coordinates as our "Bridge" keys
# bridge_keys = ["event", "layer", "tbin", "phielem", "zelem"]

# df_hit = tree_hit.arrays(bridge_keys + ["hitID", "adc"], library="pd")
# df_hit = df_hit[df_hit["adc"] > 0]

# df_trk = tree_trk.arrays(bridge_keys, library="pd")
# df_trk["is_verified"] = 1
# df_trk = df_trk.drop_duplicates(subset=bridge_keys)

# # 2. THE BRIDGE: Map the verified track flag back onto the raw truth hits
# print("Performing Spatial Inner Join (The Cluster Bridge)...")
# df_merged = pd.merge(df_hit, df_trk, on=bridge_keys, how="left")
# df_merged["is_verified"] = df_merged["is_verified"].fillna(0).astype(int)

# # 3. Analyze Track Survival Rates by hitID
# print("Calculating official tracking software rejection rates...")
# # Group by particle (hitID) per event
# track_stats = df_merged.groupby(["event", "hitID"]).agg(
#     multiplicity=("hitID", "count"),
#     verified_hits=("is_verified", "sum")
# ).reset_index()

# # We consider a truth particle "Successfully Reconstructed" if the tracking 
# # algorithm salvaged at least one of its clusters into the final ntp_clus_trk tree.
# track_stats["track_survived"] = (track_stats["verified_hits"] > 0).astype(int)

# # 4. Generate the Final Proof Report binned by Multiplicity
# survival_report = track_stats.groupby("multiplicity").agg(
#     total_truth_particles=("hitID", "count"),
#     surviving_particles=("track_survived", "sum")
# ).reset_index()

# # Calculate the rejection rate
# survival_report["rejection_rate_%"] = 100 * (1 - survival_report["surviving_particles"] / survival_report["total_truth_particles"])

# # Format for nice console printing
# print("\n" + "="*70)
# print("   ntp_clus_trk REJECTION RATE BY TRUTH MULTIPLICITY")
# print("="*70)
# print(f"{'Multiplicity (Hits)':<20} | {'Total Generated':<15} | {'Rejection Rate':<15}")
# print("-" * 70)

# # Print stats for multiplicity 1 through 6, then aggregate > 6
# for i in range(1, 130):
#     row = survival_report[survival_report["multiplicity"] == i]
#     if not row.empty:
#         total = int(row["total_truth_particles"].iloc[0])
#         rej_rate = float(row["rejection_rate_%"].iloc[0])
#         print(f" {i} hit(s) {' ':<11} | {total:<15,} | {rej_rate:>6.2f}% rejected")

# # Aggregate long, valid physics tracks (hits >= 7)
# long_tracks = survival_report[survival_report["multiplicity"] >= 130]
# total_long = long_tracks["total_truth_particles"].sum()
# survived_long = long_tracks["surviving_particles"].sum()
# rej_rate_long = 100 * (1 - survived_long / total_long) if total_long > 0 else 0

# print("-" * 70)
# print(f" >= 7 hits (Signal)  | {total_long:<15,} | {rej_rate_long:>6.2f}% rejected")
# print("=" * 70)

In [20]:
import uproot
import pandas as pd

file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
tree_hit = uproot.open(file_path)["ntp_hit"]
tree_trk = uproot.open(file_path)["ntp_clus_trk"]

bridge_keys = ["event", "layer", "tbin", "phielem", "zelem"]

# Load Raw Hits
df_hit = tree_hit.arrays(bridge_keys + ["hitID", "adc"], library="pd")
df_hit = df_hit[df_hit["adc"] > 0]

# Load found Track Clusters
df_trk = tree_trk.arrays(bridge_keys, library="pd")
df_trk["is_verified"] = 1
df_trk = df_trk.drop_duplicates(subset=bridge_keys)

print("Healing Floating-Point Mismatches and Cleaning Data...")

# --- THE FIX: Drop corrupted/null rows from our bridge keys ---
df_hit = df_hit.dropna(subset=bridge_keys)
df_trk = df_trk.dropna(subset=bridge_keys)

# 1. Cast discrete spatial grid locations to exact integers
exact_keys = ["event", "layer", "phielem", "zelem"]
df_hit[exact_keys] = df_hit[exact_keys].round().astype(int)
df_trk[exact_keys] = df_trk[exact_keys].round().astype(int)

# 2. Sort by the continuous variable (tbin) to prepare for the fuzzy join
df_hit = df_hit.sort_values("tbin")
df_trk = df_trk.sort_values("tbin")

print("Performing Fuzzy Spatial Join (The Cluster Bridge)...")
# 3. If a hit is on the same pad as a found track, and within 3 time bins
#    of the cluster's center-of-mass, we flag it as successfully reconstructed!
df_merged = pd.merge_asof(
    df_hit,
    df_trk,
    on="tbin",
    by=exact_keys,
    tolerance=3.0,
    direction="nearest"
)

df_merged["is_verified"] = df_merged["is_verified"].fillna(0).astype(int)

print("Calculating official tracking software rejection rates...")
# Group by particle (hitID) per event
track_stats = df_merged.groupby(["event", "hitID"]).agg(
    multiplicity=("hitID", "count"),
    verified_hits=("is_verified", "sum")
).reset_index()

# If the algorithm salvaged at least one cluster from this particle, it "survived"
track_stats["track_survived"] = (track_stats["verified_hits"] > 0).astype(int)

# Generate the Final Proof Report binned by Multiplicity
survival_report = track_stats.groupby("multiplicity").agg(
    total_truth_particles=("hitID", "count"),
    surviving_particles=("track_survived", "sum")
).reset_index()

survival_report["rejection_rate_%"] = 100 * (1 - survival_report["surviving_particles"] / survival_report["total_truth_particles"])

# Format for nice console printing
print("\n" + "="*70)
print("   ntp_clus_trk REJECTION RATE BY MULTIPLICITY")
print("="*70)
print(f"{'Multiplicity (Hits)':<20} | {'Total Generated':<15} | {'Rejection Rate':<15}")
print("-" * 70)

for i in range(1, 15):
    row = survival_report[survival_report["multiplicity"] == i]
    if not row.empty:
        total = int(row["total_truth_particles"].iloc[0])
        rej_rate = float(row["rejection_rate_%"].iloc[0])
        print(f" {i} hit(s) {' ':<11} | {total:<15,} | {rej_rate:>6.2f}% rejected")

# Aggregate long, valid physics tracks (hits >= 15)
long_tracks = survival_report[survival_report["multiplicity"] >= 15]
if not long_tracks.empty:
    total_long = long_tracks["total_truth_particles"].sum()
    survived_long = long_tracks["surviving_particles"].sum()
    rej_rate_long = 100 * (1 - survived_long / total_long) if total_long > 0 else 0
    print("-" * 70)
    print(f" >= 15 hits (Signal) | {total_long:<15,} | {rej_rate_long:>6.2f}% rejected")
print("=" * 70)

Healing Floating-Point Mismatches and Cleaning Data...
Performing Fuzzy Spatial Join (The Cluster Bridge)...
Calculating official tracking software rejection rates...

   ntp_clus_trk REJECTION RATE BY MULTIPLICITY
Multiplicity (Hits)  | Total Generated | Rejection Rate 
----------------------------------------------------------------------
 1 hit(s)             | 4,475,003       |  92.93% rejected
 2 hit(s)             | 2,357,867       |  90.06% rejected
 3 hit(s)             | 1,352,279       |  90.28% rejected
 4 hit(s)             | 591,214         |  87.76% rejected
 5 hit(s)             | 459,963         |  88.89% rejected
 6 hit(s)             | 277,577         |  87.28% rejected
 7 hit(s)             | 187,862         |  87.32% rejected
 8 hit(s)             | 114,612         |  85.66% rejected
 9 hit(s)             | 98,955          |  87.64% rejected
 10 hit(s)             | 61,766          |  86.04% rejected
 11 hit(s)             | 39,653          |  83.89% rejected
 12 hi

In [22]:
import uproot
import pandas as pd
import numpy as np
from scipy.spatial import KDTree

file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
tree_hit = uproot.open(file_path)["ntp_hit"]
tree_trk = uproot.open(file_path)["ntp_clus_trk"]

df_hit = tree_hit.arrays(["event", "hitID", "layer", "x", "y", "z", "adc"], library="pd")
df_hit = df_hit[df_hit["adc"] > 0].reset_index(drop=True)
df_hit["is_verified"] = 0  # Default everything to Noise

# Load found Track Clusters (The center-of-mass targets)
df_trk = tree_trk.arrays(["event", "layer", "x", "y", "z"], library="pd")

print("Bridging hits to found clusters using 3D KD-Trees...")
# We iterate by Event and Layer to ensure physical accuracy and high speed
verified_indices = []

for (ev, layer), hit_group in df_hit.groupby(['event', 'layer']):
    # Get the found cluster centers for this exact event and layer
    trk_group = df_trk[(df_trk['event'] == ev) & (df_trk['layer'] == layer)]
    
    if trk_group.empty:
        continue
        
    # Build a fast 3D spatial search tree out of the Verified Clusters
    tree = KDTree(trk_group[['x', 'y', 'z']].values)
    
    # Query all raw hits in this layer against the cluster centers.
    # Assume: If a raw pad is within 1.5 cm of a found cluster center, it is Signal.
    distances, _ = tree.query(hit_group[['x', 'y', 'z']].values, distance_upper_bound=1.5)
    
    # KDTree returns np.inf for hits outside the upper bound
    valid_mask = distances < np.inf
    
    # Save the dataframe indices of the hits that successfully mapped to a cluster
    verified_indices.extend(hit_group.index[valid_mask].tolist())

# Apply the True Signal Labels
df_hit.loc[verified_indices, 'is_verified'] = 1

print("Calculating official tracking software rejection rates...")
# Group by hitID per event
track_stats = df_hit.groupby(["event", "hitID"]).agg(
    multiplicity=("hitID", "count"),
    verified_hits=("is_verified", "sum")
).reset_index()

# We consider a truth particle "Successfully Reconstructed" if the tracking 
# algorithm salvaged at least one blob (cluster) from its trajectory.
track_stats["track_survived"] = (track_stats["verified_hits"] > 0).astype(int)

# Generate the Final Proof Report binned by Multiplicity
survival_report = track_stats.groupby("multiplicity").agg(
    total_truth_particles=("hitID", "count"),
    surviving_particles=("track_survived", "sum")
).reset_index()

survival_report["rejection_rate_%"] = 100 * (1 - survival_report["surviving_particles"] / survival_report["total_truth_particles"])

# Format for nice console printing
print("\n" + "="*70)
print("   ntp_clus_trk REJECTION RATE BY MULTIPLICITY")
print("="*70)
print(f"{'Multiplicity (Hits)':<20} | {'Total Generated':<15} | {'Rejection Rate':<15}")
print("-" * 70)

for i in range(1, 15):
    row = survival_report[survival_report["multiplicity"] == i]
    if not row.empty:
        total = int(row["total_truth_particles"].iloc[0])
        rej_rate = float(row["rejection_rate_%"].iloc[0])
        print(f" {i} hit(s) {' ':<11} | {total:<15,} | {rej_rate:>6.2f}% rejected")

# Aggregate long, valid physics tracks (hits >= 15)
long_tracks = survival_report[survival_report["multiplicity"] >= 15]
if not long_tracks.empty:
    total_long = long_tracks["total_truth_particles"].sum()
    survived_long = long_tracks["surviving_particles"].sum()
    rej_rate_long = 100 * (1 - survived_long / total_long) if total_long > 0 else 0
    print("-" * 70)
    print(f" >= 15 hits (Signal) | {total_long:<15,} | {rej_rate_long:>6.2f}% rejected")
print("=" * 70)

# Optional: Save your ML-ready Ground Truth dataset!
# df_hit.to_csv("ML_Ground_Truth_Hits.csv", index=False)

Bridging hits to found clusters using 3D KD-Trees...
Calculating official tracking software rejection rates...

   ntp_clus_trk REJECTION RATE BY MULTIPLICITY
Multiplicity (Hits)  | Total Generated | Rejection Rate 
----------------------------------------------------------------------
 1 hit(s)             | 4,503,441       |  99.68% rejected
 2 hit(s)             | 2,382,099       |  99.58% rejected
 3 hit(s)             | 1,364,808       |  99.51% rejected
 4 hit(s)             | 596,030         |  99.43% rejected
 5 hit(s)             | 461,602         |  99.34% rejected
 6 hit(s)             | 278,019         |  99.23% rejected
 7 hit(s)             | 188,014         |  99.22% rejected
 8 hit(s)             | 114,676         |  99.05% rejected
 9 hit(s)             | 98,971          |  99.04% rejected
 10 hit(s)             | 61,772          |  98.97% rejected
 11 hit(s)             | 39,660          |  98.84% rejected
 12 hit(s)             | 28,912          |  98.84% rejected
 1

In [ ]:
import uproot
import pandas as pd
import numpy as np

file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
tree_name = "ntp_clus_trk"  # The bridge tree

with uproot.open(file_path) as file:
    tree = file[tree_name]
    branches_to_extract = [
        "event",      # Event ID to group point clouds
        "seedID",     # ?
        "sR0",        # ?
        "r",          # FEATURE: Reconstructed Cluster Radius
        "phi",        # FEATURE: Reconstructed Cluster Phi
        "z"           # FEATURE: Reconstructed Cluster Z
    ]
    
    # 4. Load the data directly into a Pandas DataFrame
    print(f"Extracting {tree_name}...")
    df = tree.arrays(branches_to_extract, library="pd")

# 5. Calculate the Alignment Error (How far off the legacy Island algorithm was)
# This creates a new column showing the exact residual error
df['radial_error'] = df['r'] - df['sR0']

# Show the first few rows to validate the mapping
print(df.head())

# Optional: Save to CSV for your PyTorch DataLoader
df.to_csv("training_labels_sR0_mapped.csv", index=False)
print("Data successfully saved to training_labels_sR0_mapped.csv")

Extracting ntp_clus_trk...
   event  seedID        sR0          r       phi          z  radial_error
0    1.0     1.0  14.309492  31.500067  2.673034 -68.816292     17.190575
1    1.0     1.0  14.309492  32.065319  2.667838 -69.098061     17.755827
2    1.0     1.0  14.309492  32.635384  2.665857 -69.150826     18.325891
3    1.0     1.0  14.309492  33.199680  2.659276 -69.318817     18.890188
4    1.0     1.0  14.309492  33.767075  2.656619 -69.505638     19.457582
Data successfully saved to training_labels_sR0_mapped.csv


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("training_labels_sR0_mapped.csv")

# FIX: Only drop NaNs in the columns we actually need for geometry!
df = df.dropna(subset=['r', 'phi', 'z', 'sR0'])

# 2. Build the Two Geometries
x_reconstructed = df['r'] * np.cos(df['phi'])
y_reconstructed = df['r'] * np.sin(df['phi'])

# --- Dataset A: Cartesian ---
X_cartesian = pd.DataFrame({
    'x': x_reconstructed,
    'y': y_reconstructed,
    'z': df['z']
})

# --- Dataset B: Cylindrical (with cyclic encoding) ---
X_cylindrical = pd.DataFrame({
    'r': df['r'], 
    'sin_phi': np.sin(df['phi']), 
    'cos_phi': np.cos(df['phi']), 
    'z': df['z']
})

y_target = df['sR0']

# 3. Normalize the data (Crucial for Neural Networks)
scaler_cart = StandardScaler()
scaler_cyl = StandardScaler()

X_cart_scaled = scaler_cart.fit_transform(X_cartesian)
X_cyl_scaled = scaler_cyl.fit_transform(X_cylindrical)

# 4. Create identical Train/Test splits to ensure a fair comparison
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cart_scaled, y_target, test_size=0.2, random_state=42
)
Xcy_train, Xcy_test, ycy_train, ycy_test = train_test_split(
    X_cyl_scaled, y_target, test_size=0.2, random_state=42
)

# 5. Train Baseline Neural Networks (Multi-Layer Perceptrons)
print(f"Dataset loaded with {len(df)} hits.")
print("Training Cartesian Model...")
mlp_cart = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
mlp_cart.fit(Xc_train, yc_train)

print("Training Cylindrical Model...")
mlp_cyl = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
mlp_cyl.fit(Xcy_train, ycy_train)

# 6. Evaluate and Compare
cart_error = mean_absolute_error(yc_test, mlp_cart.predict(Xc_test))
cyl_error = mean_absolute_error(ycy_test, mlp_cyl.predict(Xcy_test))

print("\n--- RESULTS: Mean Absolute Error (cm) ---")
print(f"Cartesian Coordinate Error:   {cart_error:.4f} cm")
print(f"Cylindrical Coordinate Error: {cyl_error:.4f} cm")

# Calculate the improvement
if cyl_error < cart_error:
    improvement = ((cart_error - cyl_error) / cart_error) * 100
    print(f"\nConclusion: Cylindrical coordinates improved accuracy by {improvement:.2f}%")
else:
    print("\nConclusion: Cartesian coordinates performed better or equally well in this baseline test.")

Dataset loaded with 221494 hits.
Training Cartesian Model...
Training Cylindrical Model...

--- RESULTS: Mean Absolute Error (cm) ---
Cartesian Coordinate Error:   7.5050 cm
Cylindrical Coordinate Error: 7.4531 cm

Conclusion: Cylindrical coordinates improved accuracy by 0.69%


In [2]:
# Verify: every row in ntp_clus_trk should map to a row in ntp_cluster
# via the bridge keys (event, phielem, zelem, layer, tbin).
# If the match rate is >= 99%, treat ntp_clus_trk as a strict subset of ntp_cluster.
import uproot
import pandas as pd
import numpy as np

file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
key_cols  = ["event", "phielem", "zelem", "layer", "tbin"]

with uproot.open(file_path) as f:
    df_clus = f["ntp_cluster"].arrays(key_cols, library="pd")
    df_trk  = f["ntp_clus_trk"].arrays(key_cols, library="pd")

print(f"ntp_cluster  rows: {len(df_clus):>10,}")
print(f"ntp_clus_trk rows: {len(df_trk):>10,}   ({len(df_trk)/len(df_clus):.2%} of ntp_cluster)")

# event/phielem/zelem/layer are pad indices written as Float_t -> cast to int for exact joins.
# tbin is a float centroid; both trees should hold bit-identical values for the same cluster.
df_clus = df_clus.dropna(subset=key_cols)
df_trk  = df_trk.dropna(subset=key_cols)
for c in ("event", "phielem", "zelem", "layer"):
    df_clus[c] = df_clus[c].round().astype(np.int64)
    df_trk[c]  = df_trk[c].round().astype(np.int64)

# Left-merge: every ntp_clus_trk row should find a partner in ntp_cluster.
merged = df_trk.merge(
    df_clus.assign(_in_cluster=1),
    on=key_cols, how="left", indicator=True
)
n_total   = len(merged)
n_matched = int((merged["_merge"] == "both").sum())
n_missing = int((merged["_merge"] == "left_only").sum())

print("\nMatch summary (ntp_clus_trk -> ntp_cluster):")
print(f"  matched : {n_matched:>10,}  ({n_matched/n_total:.4%})")
print(f"  missing : {n_missing:>10,}  ({n_missing/n_total:.4%})")

# Sanity: how unique are the bridge keys inside each tree? Duplicates make the
# join one-to-many and inflate the matched count, so we want this to be small.
dup_clus = int(df_clus.duplicated(subset=key_cols, keep=False).sum())
dup_trk  = int(df_trk.duplicated(subset=key_cols,  keep=False).sum())
print("\nRows sharing the same bridge-key tuple inside each tree:")
print(f"  ntp_cluster  : {dup_clus:>10,}  ({dup_clus/len(df_clus):.4%})")
print(f"  ntp_clus_trk : {dup_trk:>10,}  ({dup_trk/len(df_trk):.4%})")

# Show a few unmatched rows for diagnosis, if any.
if n_missing:
    print("\nFirst unmatched ntp_clus_trk rows:")
    print(merged.loc[merged["_merge"] == "left_only", key_cols].head())

# Verdict you can rely on downstream.
ok = (n_missing / n_total) <= 0.01
print(f"\nVerdict: ntp_clus_trk is{'' if ok else ' NOT'} a strict subset of ntp_cluster "
      f"at the >=99% level.")


ntp_cluster  rows:  3,136,006
ntp_clus_trk rows:    221,494   (7.06% of ntp_cluster)

Match summary (ntp_clus_trk -> ntp_cluster):
  matched :    221,525  (100.0000%)
  missing :          0  (0.0000%)

Rows sharing the same bridge-key tuple inside each tree:
  ntp_cluster  :    199,107  (6.3491%)
  ntp_clus_trk :      1,359  (0.6136%)

Verdict: ntp_clus_trk is a strict subset of ntp_cluster at the >=99% level.
